# Module 5 — Notebook 5: Gaussian States, Symplectic Transformations & Williamson's Theorem

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![NumPy](https://img.shields.io/badge/numpy-2.0+-green.svg)](https://numpy.org/)
[![SciPy](https://img.shields.io/badge/scipy-1.12+-teal.svg)](https://scipy.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Publication Quality](https://img.shields.io/badge/Visualizations-Publication%20Quality-indigo.svg)]()

---

### Executive Overview & Abstract

In continuous-variable (CV) quantum optics and quantum information, **Gaussian states** form the cornerstone of theoretical analysis, experimental protocols, and resource theories. Defined by Gaussian characteristic and Wigner functions, they are completely specified by their **first moments** (displacement vector $\vec{d} \in \mathbb{R}^{2N}$) and **second moments** (covariance matrix $\mathbf{V} \in \mathbb{R}^{2N \times 2N}$). This remarkable reduction in degrees of freedom—from an infinite-dimensional Hilbert space density operator $\hat{\rho}$ down to a $2N \times 2N$ real symmetric matrix—renders CV quantum information tractable and computationally exact.

Unitary evolutions generated by Hamiltonians quadratic in the quadrature operators $(\hat{x}_k, \hat{p}_k)$ preserve the Gaussian character of quantum states. At the algebraic level, these correspond to the **real symplectic group** $\text{Sp}(2N, \mathbb{R})$, preserving the canonical commutation structure $\mathbf{\Omega}$. 

This notebook develops a rigorous, native NumPy/SciPy computational framework for multi-mode Gaussian states. We implement:
1. Construction of arbitrary multi-mode covariance matrices (vacuum, thermal, single-mode squeezed, coherent, beam-splitter mixed, and two-mode squeezed vacuum).
2. Symplectic operations: phase rotations $\mathbf{R}(\theta)$, single-mode squeezers $\mathbf{S}_1(r)$, beam splitters $\mathbf{B}(\theta)$, and two-mode squeezers $\mathbf{S}_2(r)$.
3. **Williamson's Theorem**: Diagonalization of arbitrary covariance matrices $\mathbf{V} = \mathbf{S} \mathbf{D} \mathbf{S}^T$ into thermal modes via an unconditionally stable Hermitian eigensystem formulation.
4. Physicality verification via the Heisenberg-Robertson-Schrödinger uncertainty relation: $\mathbf{V} + \frac{i}{2}\mathbf{\Omega} \ge 0$.
5. Exact evaluation of purity $\gamma$, von Neumann entropy $S$, and continuous-variable entanglement through partial transposition and **Logarithmic Negativity** $E_N$.

---

### Pedagogical & Computational Roadmap

1. **Quadrature Geometry & Symplectic Algebra**: Establish canonical conventions ($\hbar=1$, $[\hat{x}_j, \hat{p}_k] = i\delta_{jk}$, vacuum variance $\sigma_0^2 = 1/2$).
2. **Computational Engine (`GaussianStateEngine`)**: Build a modular, vectorised class for symplectic transformations, Williamson decomposition, purity, entropy, and entanglement measures.
3. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: Symplectic eigenvalues & the uncertainty principle boundary across thermal and squeezed manifolds.
   - *Panel (b)*: Entanglement quantification: Logarithmic Negativity $E_N$ vs. squeezing parameter $r$ in pure and thermal-noise environments.
   - *Panel (c)*: Covariance matrix block topology: Separable product state vs. entangled EPR state.
   - *Panel (d)*: 2D marginal Wigner phase-space contours under local vs. global correlated transformations.
4. **Physical Diagnostics & Interactive Exploration**: Explore thermal noise degradation and entanglement sudden death in Gaussian channels.
5. **Rigorous Automated Benchmark Suite**: Assert symplectic preservation ($\mathbf{S}\mathbf{\Omega}\mathbf{S}^T = \mathbf{\Omega}$), Williamson thermal reduction, and exact logarithmic negativity $E_N = 2r$.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "image.cmap": "magma",
    "lines.linewidth": 2.0,
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
})

print(f"NumPy version: {np.__version__}")
print("Symplectic Gaussian environment initialized successfully.")


---

## Theoretical Deep Dive: Symplectic Geometry & Gaussian Quantum Mechanics

### 1. Quadratures and Canonical Commutation Form

Consider an $N$-mode bosonic field described by annihilation and creation operators $\hat{a}_k, \hat{a}_k^\dagger$ satisfying $[\hat{a}_j, \hat{a}_k^\dagger] = \delta_{jk}$. We define dimensionless quadrature operators with the convention $\hbar = 1$:
$$\hat{x}_k = \frac{\hat{a}_k + \hat{a}_k^\dagger}{\sqrt{2}}, \quad \hat{p}_k = \frac{\hat{a}_k - \hat{a}_k^\dagger}{i\sqrt{2}}$$
Grouping the quadratures into a vector operator:
$$\hat{\mathbf{R}} = \left(\hat{x}_1, \hat{p}_1, \hat{x}_2, \hat{p}_2, \dots, \hat{x}_N, \hat{p}_N\right)^T \in \mathbb{R}^{2N}$$
The canonical commutation relations take the compact matrix form:
$$\left[\hat{R}_j, \hat{R}_k\right] = i \Omega_{jk}, \quad \mathbf{\Omega} = \bigoplus_{k=1}^N \mathbf{\omega}_0 = \bigoplus_{k=1}^N \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}$$
where $\mathbf{\Omega}$ is the real skew-symmetric **symplectic form** satisfying $\mathbf{\Omega}^T = -\mathbf{\Omega} = \mathbf{\Omega}^{-1}$ and $\mathbf{\Omega}^2 = -\mathbf{I}_{2N}$.

---

### 2. Gaussian States and the Covariance Matrix

A quantum state $\hat{\rho}$ is **Gaussian** if its Wigner function (or characteristic function) is a multi-variate Gaussian distribution:
$$W(\mathbf{R}) = \frac{1}{\pi^N \sqrt{\det \mathbf{V}}} \exp\left( -(\mathbf{R} - \vec{d})^T \mathbf{V}^{-1} (\mathbf{R} - \vec{d}) \right)$$
where the state is uniquely defined by:
1. **Displacement vector**: $d_j = \langle \hat{R}_j \rangle = \text{Tr}(\hat{\rho} \hat{R}_j)$
2. **Covariance Matrix (CM)** $\mathbf{V} \in \mathbb{R}^{2N \times 2N}$:
   $$V_{jk} = \frac{1}{2} \left\langle \left\{ \hat{R}_j - d_j, \hat{R}_k - d_k \right\} \right\rangle = \frac{1}{2} \text{Tr}\left[ \hat{\rho} \left( (\hat{R}_j - d_j)(\hat{R}_k - d_k) + (\hat{R}_k - d_k)(\hat{R}_j - d_j) \right) \right]$$
For the vacuum state $|0\rangle^{\otimes N}$, $\mathbf{V}_{\text{vac}} = \frac{1}{2}\mathbf{I}_{2N}$.

---

### 3. The Symplectic Group $\text{Sp}(2N, \mathbb{R})$

Unitary transformations $\hat{U} = \exp\left(-\frac{i}{2} \hat{\mathbf{R}}^T \mathbf{H} \hat{\mathbf{R}}\right)$ generated by quadratic Hamiltonians induce linear transformations in phase space:
$$\hat{\mathbf{R}}' = \hat{U}^\dagger \hat{\mathbf{R}} \hat{U} = \mathbf{S} \hat{\mathbf{R}}$$
Preservation of the commutation relations $[\hat{R}_j', \hat{R}_k'] = i\Omega_{jk}$ requires:
$$\mathbf{S} \mathbf{\Omega} \mathbf{S}^T = \mathbf{\Omega}$$
Any real $2N \times 2N$ matrix satisfying this equation belongs to the **real symplectic group** $\text{Sp}(2N, \mathbb{R})$. Under $\mathbf{S}$, the covariance matrix and displacement transform via:
$$\mathbf{V}' = \mathbf{S} \mathbf{V} \mathbf{S}^T, \quad \vec{d}' = \mathbf{S} \vec{d}$$

---

### 4. Williamson's Theorem and the Symplectic Spectrum

A celebrated theorem by John Williamson (1936) states:
> For any real symmetric positive-definite $2N \times 2N$ matrix $\mathbf{V}$, there exists a symplectic transformation $\mathbf{S} \in \text{Sp}(2N, \mathbb{R})$ such that:
> $$\mathbf{V} = \mathbf{S} \mathbf{D} \mathbf{S}^T, \quad \mathbf{D} = \text{diag}(\nu_1, \nu_1, \nu_2, \nu_2, \dots, \nu_N, \nu_N)$$
> where $\nu_k \ge 1/2$ are the **symplectic eigenvalues** of $\mathbf{V}$.

Physical significance: **Every mixed Gaussian state can be transformed by linear symplectic optics into an uncorrelated product of thermal states** with mean photon numbers $\bar{n}_k = \nu_k - \frac{1}{2}$.

#### Stable Computational Method for $\nu_k$:
The symplectic spectrum $\{\nu_k\}$ can be extracted by noting that the matrix $\mathbf{M} = i\mathbf{\Omega}\mathbf{V}$ has eigenvalues $\pm \nu_k$. Equivalently and with superior numerical stability:
$$\mathbf{A} = \mathbf{V}^{1/2} (-i\mathbf{\Omega}) \mathbf{V}^{1/2}$$
$\mathbf{A}$ is explicitly **Hermitian**:
$$\mathbf{A}^\dagger = \mathbf{V}^{1/2} (i\mathbf{\Omega}^T) \mathbf{V}^{1/2} = \mathbf{V}^{1/2} (-i\mathbf{\Omega}) \mathbf{V}^{1/2} = \mathbf{A}$$
The standard Hermitian eigenvalue routine `scipy.linalg.eigvalsh(A)` yields $2N$ strictly real eigenvalues $\{\pm \nu_k\}$. Taking the positive half yields the exact symplectic spectrum.

---

### 5. Physicality, Purity, Entropy & Entanglement

- **Heisenberg-Robertson Uncertainty Relation**:
  $$\mathbf{V} + \frac{i}{2}\mathbf{\Omega} \ge 0 \iff \nu_k \ge \frac{1}{2} \quad \forall k=1, \dots, N$$
- **Purity**:
  $$\gamma = \text{Tr}(\hat{\rho}^2) = \frac{1}{2^N \sqrt{\det \mathbf{V}}} = \prod_{k=1}^N \frac{1}{2\nu_k} \le 1$$
  The state is pure if and only if $\nu_k = 1/2$ for all $k$, which implies $\det \mathbf{V} = 1/4^N$.
- **von Neumann Entropy**:
  $$S(\hat{\rho}) = \sum_{k=1}^N f(\nu_k), \quad f(x) = \left(x + \frac{1}{2}\right)\ln\left(x + \frac{1}{2}\right) - \left(x - \frac{1}{2}\right)\ln\left(x - \frac{1}{2}\right)$$
- **Two-Mode Entanglement & Simon's Criterion**:
  For a two-mode Gaussian state partitioned into modes $A$ and $B$, partial transposition (time-reversal on mode $B$: $\hat{p}_2 \to -\hat{p}_2$) corresponds to $\tilde{\mathbf{V}} = \mathbf{\Lambda} \mathbf{V} \mathbf{\Lambda}$, where $\mathbf{\Lambda} = \text{diag}(1, 1, 1, -1)$.
  The state is separable if and only if:
  $$\tilde{\mathbf{V}} + \frac{i}{2}\mathbf{\Omega} \ge 0 \iff \tilde{\nu}_- \ge \frac{1}{2}$$
  where $\tilde{\nu}_-$ is the smallest symplectic eigenvalue of $\tilde{\mathbf{V}}$.
  The **Logarithmic Negativity** quantifies the distillable entanglement:
  $$E_N = \max\left(0, -\ln(2\tilde{\nu}_-)\right)$$
  For a pure Two-Mode Squeezed Vacuum (TMSV) state with squeezing $r$, $\tilde{\nu}_- = \frac{1}{2}e^{-2r}$, yielding the exact linear identity:
  $$E_N = 2r$$


In [ ]:
# Cell 4: The Gaussian State Computational Engine
class GaussianStateEngine:
    '''
    Publication-grade computational engine for continuous-variable Gaussian states,
    symplectic operations, Williamson decomposition, and entanglement measures.
    '''

    @staticmethod
    def symplectic_form(num_modes: int) -> np.ndarray:
        '''Constructs the 2N x 2N symplectic form Omega = \bigoplus_{k=1}^N [[0, 1], [-1, 0]].'''
        omega_0 = np.array([[0.0, 1.0], [-1.0, 0.0]])
        blocks = [omega_0 for _ in range(num_modes)]
        return la.block_diag(*blocks)

    @classmethod
    def is_symplectic(cls, S: np.ndarray, tol: float = 1e-10) -> bool:
        '''Verifies if matrix S satisfies S Omega S^T = Omega.'''
        num_modes = S.shape[0] // 2
        Omega = cls.symplectic_form(num_modes)
        res = S @ Omega @ S.T - Omega
        return np.max(np.abs(res)) < tol

    @staticmethod
    def rotation(theta: float) -> np.ndarray:
        '''Single-mode phase-space rotation matrix by angle theta.'''
        c, s = np.cos(theta), np.sin(theta)
        return np.array([[c, s], [-s, c]])

    @staticmethod
    def single_mode_squeezer(r: float, phi: float = 0.0) -> np.ndarray:
        '''Single-mode squeezing symplectic matrix for parameter r and squeezing angle phi.'''
        R = GaussianStateEngine.rotation(phi / 2.0)
        S_diag = np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])
        return R @ S_diag @ R.T

    @staticmethod
    def beam_splitter(theta: float) -> np.ndarray:
        '''
        Two-mode beam splitter symplectic matrix with mixing angle theta:
        B(theta) = [[cos(theta)*I_2, sin(theta)*I_2], [-sin(theta)*I_2, cos(theta)*I_2]].
        '''
        c, s = np.cos(theta), np.sin(theta)
        I2 = np.eye(2)
        return np.block([[c * I2, s * I2], [-s * I2, c * I2]])

    @staticmethod
    def two_mode_squeezer(r: float) -> np.ndarray:
        '''
        Two-mode squeezing symplectic matrix with parameter r:
        S_2(r) = [[cosh(r)*I_2, sinh(r)*Z], [sinh(r)*Z, cosh(r)*I_2]] where Z = diag(1, -1).
        '''
        ch, sh = np.cosh(r), np.sinh(r)
        I2 = np.eye(2)
        Z = np.diag([1.0, -1.0])
        return np.block([[ch * I2, sh * Z], [sh * Z, ch * I2]])

    @staticmethod
    def vacuum_state(num_modes: int) -> Tuple[np.ndarray, np.ndarray]:
        '''Returns (d, V) for N-mode vacuum state (V = 0.5 * I_{2N}).'''
        d = np.zeros(2 * num_modes)
        V = 0.5 * np.eye(2 * num_modes)
        return d, V

    @staticmethod
    def thermal_state(num_modes: int, n_th: Union[float, List[float]]) -> Tuple[np.ndarray, np.ndarray]:
        '''Returns (d, V) for N-mode thermal state with average photon number(s) n_th.'''
        if np.isscalar(n_th):
            n_th = [float(n_th)] * num_modes
        d = np.zeros(2 * num_modes)
        blocks = [(n + 0.5) * np.eye(2) for n in n_th]
        V = la.block_diag(*blocks)
        return d, V

    @staticmethod
    def two_mode_squeezed_vacuum(r: float) -> Tuple[np.ndarray, np.ndarray]:
        '''Returns (d, V) for pure two-mode squeezed vacuum (TMSV) state.'''
        d, V_vac = GaussianStateEngine.vacuum_state(2)
        S = GaussianStateEngine.two_mode_squeezer(r)
        V_tmsv = S @ V_vac @ S.T
        return d, V_tmsv

    @classmethod
    def symplectic_spectrum(cls, V: np.ndarray) -> np.ndarray:
        '''
        Computes the symplectic eigenvalues {nu_k} of a 2N x 2N covariance matrix V.
        Uses the Hermitian matrix A = V^{1/2} (-i Omega) V^{1/2} for unconditional stability.
        '''
        num_modes = V.shape[0] // 2
        Omega = cls.symplectic_form(num_modes)
        
        # Compute matrix square root V^{1/2}
        evals_V, evecs_V = la.eigh(V)
        if np.any(evals_V <= 0):
            # Matrix is not strictly positive-definite
            raise ValueError("Covariance matrix must be strictly positive-definite.")
        V_sqrt = evecs_V @ np.diag(np.sqrt(evals_V)) @ evecs_V.T
        
        # Construct Hermitian matrix A = V^{1/2} (-i Omega) V^{1/2}
        A = V_sqrt @ (-1.0j * Omega) @ V_sqrt
        # Eigenvalues of A come in +/- nu_k pairs
        eigvals_A = la.eigvalsh(A)
        # Extract the positive eigenvalues and sort ascending
        pos_eigs = np.sort(eigvals_A[eigvals_A > 0])
        if len(pos_eigs) != num_modes:
            # Handle potential degeneracy/numerical precision edge cases
            pos_eigs = np.sort(np.abs(eigvals_A))[num_modes:]
        return pos_eigs

    @classmethod
    def is_physical(cls, V: np.ndarray, tol: float = 1e-10) -> bool:
        '''Verifies the Heisenberg-Robertson uncertainty relation: nu_k >= 0.5 - tol.'''
        nus = cls.symplectic_spectrum(V)
        return np.all(nus >= 0.5 - tol)

    @classmethod
    def purity(cls, V: np.ndarray) -> float:
        '''Computes purity gamma = 1 / (2^N * sqrt(det(V))) = prod_k (1 / (2 * nu_k)).'''
        num_modes = V.shape[0] // 2
        det_V = la.det(V)
        if det_V <= 0:
            return 0.0
        return float(1.0 / ((2.0 ** num_modes) * np.sqrt(det_V)))

    @classmethod
    def von_neumann_entropy(cls, V: np.ndarray) -> float:
        '''Computes von Neumann entropy S = sum_k f(nu_k) from symplectic eigenvalues.'''
        nus = cls.symplectic_spectrum(V)
        entropy = 0.0
        for nu in nus:
            if nu > 0.5 + 1e-14:
                plus = nu + 0.5
                minus = nu - 0.5
                entropy += plus * np.log(plus) - minus * np.log(minus)
        return float(entropy)

    @classmethod
    def logarithmic_negativity(cls, V: np.ndarray) -> float:
        '''
        Computes the logarithmic negativity E_N for a two-mode Gaussian state.
        E_N = max(0, -ln(2 * \tilde{\nu}_-)) where \tilde{\nu}_- is the smallest symplectic eigenvalue of \tilde{V}.
        '''
        assert V.shape == (4, 4), "Logarithmic negativity implemented for two-mode systems (4x4 CM)."
        # Partial transposition: p_2 -> -p_2
        Lambda = np.diag([1.0, 1.0, 1.0, -1.0])
        V_pt = Lambda @ V @ Lambda
        nus_pt = cls.symplectic_spectrum(V_pt)
        nu_min = np.min(nus_pt)
        if nu_min < 0.5:
            return float(-np.log(2.0 * nu_min))
        return 0.0

    @staticmethod
    def wigner_2d(V: np.ndarray, d: np.ndarray, x_grid: np.ndarray, p_grid: np.ndarray) -> np.ndarray:
        '''Evaluates the 2D Wigner function on a spatial grid for a single-mode state.'''
        assert V.shape == (2, 2), "Wigner evaluation requires 2x2 single-mode covariance matrix."
        X, P = np.meshgrid(x_grid, p_grid)
        det_V = la.det(V)
        V_inv = la.inv(V)
        
        dX = X - d[0]
        dP = P - d[1]
        
        quad_form = V_inv[0, 0] * dX**2 + 2.0 * V_inv[0, 1] * dX * dP + V_inv[1, 1] * dP**2
        W = (1.0 / (np.pi * np.sqrt(det_V))) * np.exp(-quad_form)
        return W

print("GaussianStateEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Quality 4-Panel Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Gaussian States, Symplectic Transformations & Williamson's Theorem", fontsize=16, y=0.98)

# -----------------------------------------------------------------
# Panel (a): Symplectic Spectrum and Physicality Boundary
# -----------------------------------------------------------------
ax_a = axes[0, 0]
n_th_vals = np.linspace(0.0, 2.0, 100)
r_fixed = 0.8

# Compare pure thermal vs thermal + squeezed states
nu_thermal = []
nu_squeezed_thermal = []

for n in n_th_vals:
    # 1-mode thermal
    _, V_th = GaussianStateEngine.thermal_state(1, n)
    nu_thermal.append(GaussianStateEngine.symplectic_spectrum(V_th)[0])
    
    # 1-mode squeezed thermal
    S1 = GaussianStateEngine.single_mode_squeezer(r_fixed)
    V_sq_th = S1 @ V_th @ S1.T
    nu_squeezed_thermal.append(GaussianStateEngine.symplectic_spectrum(V_sq_th)[0])

ax_a.plot(n_th_vals, nu_thermal, label=r"Thermal State ($\nu = \bar{n} + 1/2$)", color="#1f77b4", lw=2.5)
ax_a.plot(n_th_vals, nu_squeezed_thermal, "--", label=rf"Squeezed Thermal ($r = {r_fixed}$, $\nu = \bar{n} + 1/2$)", color="#ff7f0e", lw=2.5)
ax_a.axhline(0.5, color="red", linestyle=":", lw=2.0, label=r"Quantum Physicality Bound ($\nu \geq 1/2$)")
ax_a.fill_between(n_th_vals, 0.0, 0.5, color="red", alpha=0.15, label="Unphysical Region")

ax_a.set_title(r"(a) Symplectic Invariance: $\nu = \bar{n}_{\mathrm{th}} + 1/2$", pad=10)
ax_a.set_xlabel(r"Thermal Photon Number $\bar{n}_{\mathrm{th}}$")
ax_a.set_ylabel(r"Symplectic Eigenvalue $\nu$")
ax_a.set_xlim(0, 2.0)
ax_a.set_ylim(0, 2.7)
ax_a.legend(loc="upper left", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (b): Entanglement & Logarithmic Negativity vs Squeezing
# -----------------------------------------------------------------
ax_b = axes[0, 1]
r_scan = np.linspace(0.0, 1.8, 100)
E_N_pure = []
E_N_noisy_02 = []
E_N_noisy_05 = []

for r in r_scan:
    # Pure TMSV
    _, V_tmsv = GaussianStateEngine.two_mode_squeezed_vacuum(r)
    E_N_pure.append(GaussianStateEngine.logarithmic_negativity(V_tmsv))
    
    # TMSV with thermal background n_th = 0.2
    _, V_th_02 = GaussianStateEngine.thermal_state(2, 0.2)
    S2 = GaussianStateEngine.two_mode_squeezer(r)
    V_noisy_02 = S2 @ V_th_02 @ S2.T
    E_N_noisy_02.append(GaussianStateEngine.logarithmic_negativity(V_noisy_02))
    
    # TMSV with thermal background n_th = 0.5
    _, V_th_05 = GaussianStateEngine.thermal_state(2, 0.5)
    V_noisy_05 = S2 @ V_th_05 @ S2.T
    E_N_noisy_05.append(GaussianStateEngine.logarithmic_negativity(V_noisy_05))

ax_b.plot(r_scan, E_N_pure, label=r"Pure TMSV ($E_N = 2r$)", color="#2ca02c", lw=2.5)
ax_b.plot(r_scan, E_N_noisy_02, label=r"Thermal TMSV ($\bar{n}_{\mathrm{th}} = 0.2$)", color="#9467bd", lw=2.0)
ax_b.plot(r_scan, E_N_noisy_05, label=r"Thermal TMSV ($\bar{n}_{\mathrm{th}} = 0.5$)", color="#8c564b", lw=2.0)
ax_b.axhline(0.0, color="gray", linestyle="-", lw=1.0)

ax_b.set_title(r"(b) Continuous-Variable Entanglement $E_N$", pad=10)
ax_b.set_xlabel(r"Squeezing Parameter $r$")
ax_b.set_ylabel(r"Logarithmic Negativity $E_N$")
ax_b.set_xlim(0, 1.8)
ax_b.set_ylim(-0.1, 3.8)
ax_b.legend(loc="upper left", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (c): Covariance Matrix Structure: Separable vs Entangled
# -----------------------------------------------------------------
ax_c = axes[1, 0]
_, V_epr = GaussianStateEngine.two_mode_squeezed_vacuum(r=1.0)
im = ax_c.imshow(V_epr, cmap="coolwarm", vmin=-np.max(V_epr), vmax=np.max(V_epr))

# Labels for quadratures
labels = [r"$x_1$", r"$p_1$", r"$x_2$", r"$p_2$"]
ax_c.set_xticks(range(4))
ax_c.set_yticks(range(4))
ax_c.set_xticklabels(labels)
ax_c.set_yticklabels(labels)

# Overlay numerical values inside cells
for i in range(4):
    for j in range(4):
        val = V_epr[i, j]
        color = "white" if abs(val) > 1.0 else "black"
        ax_c.text(j, i, f"{val:+.2f}", ha="center", va="center", color=color, fontweight="bold")

# Block outlines separating modes
ax_c.axvline(1.5, color="black", lw=2.0)
ax_c.axhline(1.5, color="black", lw=2.0)

ax_c.set_title(r"(c) EPR Covariance Matrix $\mathbf{V}_{\mathrm{EPR}}$ ($r=1.0$)", pad=10)
fig.colorbar(im, ax=ax_c, fraction=0.046, pad=0.04, label="Covariance Element")

# -----------------------------------------------------------------
# Panel (d): Marginal Wigner Phase-Space Contours
# -----------------------------------------------------------------
ax_d = axes[1, 1]
grid = np.linspace(-3.0, 3.0, 150)
d_vac, V_vac = GaussianStateEngine.vacuum_state(1)
W_vac = GaussianStateEngine.wigner_2d(V_vac, d_vac, grid, grid)

# Reduced state of mode 1 from TMSV
V_mode1 = V_epr[:2, :2]
d_mode1 = np.zeros(2)
W_mode1 = GaussianStateEngine.wigner_2d(V_mode1, d_mode1, grid, grid)

# Squeezed state (single mode)
S1 = GaussianStateEngine.single_mode_squeezer(r=0.8)
V_sq = S1 @ V_vac @ S1.T
W_sq = GaussianStateEngine.wigner_2d(V_sq, d_vac, grid, grid)

# Plot contour levels for 1-sigma uncertainty ellipses
ax_d.contour(grid, grid, W_vac, levels=[0.1], colors=["#1f77b4"], linewidths=2.5)
ax_d.contour(grid, grid, W_sq, levels=[0.1], colors=["#d62728"], linewidths=2.5)
ax_d.contour(grid, grid, W_mode1, levels=[0.05], colors=["#2ca02c"], linewidths=2.5)

# Custom legend proxies
from matplotlib.lines import Line2D
proxies = [
    Line2D([0], [0], color="#1f77b4", lw=2.5, label="Vacuum State (Circular, Pure)"),
    Line2D([0], [0], color="#d62728", lw=2.5, label=r"Single-Mode Squeezed ($r=0.8$, Pure)"),
    Line2D([0], [0], color="#2ca02c", lw=2.5, label=r"EPR Mode 1 (Thermal Mixed, $\gamma < 1$)"),
]
ax_d.legend(handles=proxies, loc="upper right", framealpha=0.9)

ax_d.set_title(r"(d) Marginal Phase-Space Wigner Contours", pad=10)
ax_d.set_xlabel(r"Position Quadrature $x$")
ax_d.set_ylabel(r"Momentum Quadrature $p$")
ax_d.set_xlim(-3.0, 3.0)
ax_d.set_ylim(-3.0, 3.0)
ax_d.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax_d.axhline(0, color="gray", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()


---

## Physical Interpretation & Detailed Mathematical Insights

### 1. Invariance of the Symplectic Spectrum under Symplectic Operations

Panel (a) illustrates a profound theorem in continuous-variable physics:
> **The symplectic eigenvalues $\{\nu_k\}$ are invariant under all symplectic transformations $\mathbf{S} \in \text{Sp}(2N, \mathbb{R})$.**

When a single-mode thermal state with variance $\mathbf{V}_{\text{th}} = (\bar{n} + 1/2)\mathbf{I}_2$ is squeezed by $\mathbf{S}_1(r)$, its quadrature variances deform to:
$$\Delta x^2 = e^{-2r}\left(\bar{n} + \frac{1}{2}\right), \quad \Delta p^2 = e^{2r}\left(\bar{n} + \frac{1}{2}\right)$$
Although the state is strongly squeezed in position and anti-squeezed in momentum, its determinant is unchanged:
$$\det \mathbf{V}_{\text{sq}} = \Delta x^2 \Delta p^2 = \left(\bar{n} + \frac{1}{2}\right)^2 \implies \nu = \bar{n} + \frac{1}{2}$$
The symplectic eigenvalue measures the **intrinsic quantum thermal entropy** of the state, independent of any coordinate rotation or squeezing applied by linear optics. It represents the "mixedness" that cannot be removed by unitary Gaussian operations.

---

### 2. Entanglement Genesis and Thermal Vulnerability

Panel (b) examines the creation of continuous-variable entanglement via the two-mode squeezer $\hat{S}_2(r)$.
- **Pure TMSV**: For pure vacuum input, the logarithmic negativity follows the exact relation:
  $$E_N(r) = 2r$$
  Every increment in squeezing parameter $r$ adds $2$ ebits of logarithmic negativity.
- **Thermal Mixed Input**: When the input modes have thermal photon occupation $\bar{n}_{\text{th}} > 0$, the state starts with symplectic eigenvalues $\nu_0 = \bar{n}_{\text{th}} + 1/2 > 1/2$.
  Under partial transposition:
  $$\tilde{\nu}_- = \left(\bar{n}_{\text{th}} + \frac{1}{2}\right) e^{-2r}$$
  For entanglement to exist ($E_N > 0$), we strictly require $\tilde{\nu}_- < 1/2$, which yields a **critical squeezing threshold**:
  $$e^{2r_{\mathrm{crit}}} = 2\bar{n}_{\text{th}} + 1 \implies r_{\mathrm{crit}} = \frac{1}{2} \ln(2\bar{n}_{\text{th}} + 1)$$
  Below $r_{\mathrm{crit}}$, the state is strictly **separable** ($E_N \equiv 0$) despite being non-trivially correlated! This demonstrates the sensitivity of continuous-variable entanglement to environmental thermal noise.

---

### 3. Covariance Matrix Geometry: The Anatomy of an EPR State

Panel (c) displays the covariance matrix of an EPR state ($r=1.0$):
$$\mathbf{V}_{\mathrm{EPR}} = \frac{1}{2} \begin{pmatrix} 
\cosh(2r) & 0 & \sinh(2r) & 0 \\
0 & \cosh(2r) & 0 & -\sinh(2r) \\
\sinh(2r) & 0 & \cosh(2r) & 0 \\
0 & -\sinh(2r) & 0 & \cosh(2r)
\end{pmatrix}$$
- **Diagonal blocks $\mathbf{A} = \mathbf{B} = \frac{1}{2}\cosh(2r)\mathbf{I}_2$**: Describe local thermal states with variance $\frac{1}{2}\cosh(2) \approx 1.88$ ($\bar{n} \approx 1.38$).
- **Off-diagonal block $\mathbf{C} = \frac{1}{2}\sinh(2r)\mathbf{Z}$**: Encodes quantum correlations. Notice the crucial sign change:
  - $V_{x_1, x_2} = +\frac{1}{2}\sinh(2r) > 0$ (strong position correlation: $x_1 \approx x_2$)
  - $V_{p_1, p_2} = -\frac{1}{2}\sinh(2r) < 0$ (strong momentum anti-correlation: $p_1 \approx -p_2$)
This cross-block structure is the exact continuous-variable manifestation of the Einstein-Podolsky-Rosen paradox.


In [ ]:
# Cell 7: Interactive Diagnostic Suite for Gaussian States
def analyze_two_mode_gaussian(
    r_squeeze: float = 1.0, 
    theta_bs: float = np.pi / 4.0, 
    n_thermal: float = 0.0
) -> Dict[str, float]:
    '''
    Performs full symplectic, purity, entropy, and entanglement diagnostics
    on a two-mode Gaussian state prepared by squeezing and beam-splitter mixing.
    '''
    # Step 1: Prepare thermal input
    d0, V0 = GaussianStateEngine.thermal_state(2, n_thermal)
    
    # Step 2: Apply single-mode squeezing on mode 1
    S1 = GaussianStateEngine.single_mode_squeezer(r_squeeze)
    S_step2 = la.block_diag(S1, np.eye(2))
    V1 = S_step2 @ V0 @ S_step2.T
    
    # Step 3: Apply beam splitter
    B = GaussianStateEngine.beam_splitter(theta_bs)
    V_final = B @ V1 @ B.T
    
    # Step 4: Extract physical invariants
    is_phys = GaussianStateEngine.is_physical(V_final)
    symp_spec = GaussianStateEngine.symplectic_spectrum(V_final)
    purity_val = GaussianStateEngine.purity(V_final)
    entropy_val = GaussianStateEngine.von_neumann_entropy(V_final)
    entanglement_val = GaussianStateEngine.logarithmic_negativity(V_final)
    
    results = {
        "is_physical": is_phys,
        "nu_1": symp_spec[0],
        "nu_2": symp_spec[1],
        "purity": purity_val,
        "von_neumann_entropy": entropy_val,
        "logarithmic_negativity": entanglement_val,
    }
    
    print("=" * 65)
    print("GAUSSIAN STATE DIAGNOSTIC REPORT")
    print("=" * 65)
    print(f"Input Parameters: r = {r_squeeze:.2f}, theta_BS = {theta_bs/np.pi:.2f} pi, n_th = {n_thermal:.2f}")
    print(f"- Physically Valid:          {is_phys} (nu_min = {np.min(symp_spec):.4f} >= 0.5)")
    print(f"- Symplectic Spectrum:       nu_1 = {symp_spec[0]:.4f}, nu_2 = {symp_spec[1]:.4f}")
    print(f"- State Purity:              gamma = {purity_val:.4f} (Pure = 1.0000)")
    print(f"- von Neumann Entropy:       S = {entropy_val:.4f} nats")
    print(f"- Entanglement (Log-Neg):    E_N = {entanglement_val:.4f} ebits")
    print("=" * 65)
    return results

# Example evaluation
diag_report = analyze_two_mode_gaussian(r_squeeze=1.0, theta_bs=np.pi / 4.0, n_thermal=0.0)


---

## Guided Exercises & Continuous-Variable Quantum Challenges

### Challenge 1: The Serafini Invariants & Simon's Separability Criterion
For any two-mode covariance matrix written in $2 \times 2$ block form:
$$\mathbf{V} = \begin{pmatrix} \mathbf{A} & \mathbf{C} \\ \mathbf{C}^T & \mathbf{B} \end{pmatrix}$$
The four local symplectic invariants are defined as:
$$I_1 = \det \mathbf{A}, \quad I_2 = \det \mathbf{B}, \quad I_3 = \det \mathbf{C}, \quad I_4 = \det \mathbf{V}$$
**Analytical Task**: Express the symplectic eigenvalues $\nu_\pm$ of $\mathbf{V}$ directly in terms of $\Delta(\mathbf{V}) = I_1 + I_2 + 2I_3$ and $I_4$:
$$2\nu_\pm^2 = \Delta(\mathbf{V}) \pm \sqrt{\Delta(\mathbf{V})^2 - 4I_4}$$
Prove that under partial transposition $\mathbf{C} \to \mathbf{C} \mathbf{Z}$, $I_3 \to -I_3$, so $\tilde{\Delta} = I_1 + I_2 - 2I_3$. Hence derive Simon's exact determinantal separability condition:
$$I_1 I_2 - I_4 \ge \frac{1}{16} + \frac{1}{4}(I_1 + I_2 - 2I_3)$$

---

### Challenge 2: Passive Linear Networks as the Compact Subgroup $K(N)$
Any passive linear optical interferometer composed entirely of beam splitters and phase shifters conserves total photon number $\sum_k \hat{a}_k^\dagger \hat{a}_k$.
**Computational Task**:
1. Show that the set of symplectic transformations that are also orthogonal matrices:
   $$K(N) = \text{Sp}(2N, \mathbb{R}) \cap O(2N)$$
   forms a maximal compact subgroup isomorphic to the unitary group $U(N)$.
2. Verify that for any $\mathbf{U} \in K(N)$, $\mathbf{U}^T \mathbf{U} = \mathbf{I}_{2N}$ and $\mathbf{U}\mathbf{\Omega}\mathbf{U}^T = \mathbf{\Omega}$.
3. Demonstrate that such transformations preserve the total variance $\text{Tr}(\mathbf{V})$.

---

### Challenge 3: Entanglement Degradability in a Gaussian Loss Channel
Consider an entangled EPR pair where Mode $B$ is transmitted through a lossy optical fiber modeled as a beam splitter with transmission $\eta \in [0, 1]$ coupled to a vacuum environment:
$$\mathbf{V}_B' = \eta \mathbf{V}_B + (1 - \eta) \frac{1}{2}\mathbf{I}_2, \quad \mathbf{C}' = \sqrt{\eta} \mathbf{C}$$
**Computational Task**:
1. Implement the loss channel map $\mathcal{E}_\eta(\mathbf{V})$.
2. Determine whether continuous-variable entanglement experiences **entanglement sudden death** (finite $\eta > 0$ where $E_N = 0$) or if $E_N > 0$ for all $\eta > 0$ when coupled to pure vacuum.
3. Contrast this with transmission through a thermal environment with $\bar{n}_{\mathrm{env}} > 0$.


In [ ]:
# Cell 9: Automated Benchmark & Verification Suite
def run_benchmark_suite():
    print("=" * 70)
    print("STARTING GAUSSIAN STATES & SYMPLECTIC BENCHMARK SUITE")
    print("=" * 70)
    
    # -----------------------------------------------------------------
    # Test 1: Symplectic Group Property Preservation
    # -----------------------------------------------------------------
    print("Test 1: Verifying Sp(2N, R) group relations for standard optical elements...")
    S_rot = GaussianStateEngine.rotation(0.73)
    S_sq1 = GaussianStateEngine.single_mode_squeezer(1.2, 0.4)
    S_bs = GaussianStateEngine.beam_splitter(np.pi / 3.0)
    S_sq2 = GaussianStateEngine.two_mode_squeezer(0.85)
    
    assert GaussianStateEngine.is_symplectic(S_rot), "Single-mode rotation failed symplectic condition!"
    assert GaussianStateEngine.is_symplectic(S_sq1), "Single-mode squeezer failed symplectic condition!"
    assert GaussianStateEngine.is_symplectic(S_bs), "Beam splitter failed symplectic condition!"
    assert GaussianStateEngine.is_symplectic(S_sq2), "Two-mode squeezer failed symplectic condition!"
    print("  -> All optical transformations strictly preserve Omega (PASS)")

    # -----------------------------------------------------------------
    # Test 2: Williamson Decomposition of Pure Vacuum and Squeezed States
    # -----------------------------------------------------------------
    print("\nTest 2: Verifying Williamson symplectic spectrum of pure states...")
    _, V_vac = GaussianStateEngine.vacuum_state(3)
    nus_vac = GaussianStateEngine.symplectic_spectrum(V_vac)
    assert np.allclose(nus_vac, 0.5, atol=1e-12), f"Vacuum spectrum mismatch: {nus_vac}"
    
    _, V_tmsv = GaussianStateEngine.two_mode_squeezed_vacuum(r=1.3)
    nus_tmsv = GaussianStateEngine.symplectic_spectrum(V_tmsv)
    assert np.allclose(nus_tmsv, 0.5, atol=1e-12), f"Pure TMSV spectrum mismatch: {nus_tmsv}"
    
    purity_tmsv = GaussianStateEngine.purity(V_tmsv)
    assert np.isclose(purity_tmsv, 1.0, atol=1e-12), f"Pure TMSV purity mismatch: {purity_tmsv}"
    print(f"  -> Vacuum and TMSV symplectic spectrum nu = {nus_vac[0]:.4f} == 0.5 (PASS)")
    print(f"  -> Pure TMSV purity gamma = {purity_tmsv:.4f} == 1.0 (PASS)")

    # -----------------------------------------------------------------
    # Test 3: Exact Logarithmic Negativity Identity E_N = 2r
    # -----------------------------------------------------------------
    print("\nTest 3: Verifying Logarithmic Negativity exact linear identity E_N = 2r...")
    test_r_vals = [0.2, 0.5, 0.8, 1.2, 1.5]
    for r in test_r_vals:
        _, V = GaussianStateEngine.two_mode_squeezed_vacuum(r)
        E_N = GaussianStateEngine.logarithmic_negativity(V)
        expected_EN = 2.0 * r
        rel_err = abs(E_N - expected_EN) / expected_EN
        assert rel_err < 1e-10, f"Log-negativity failed for r={r}: {E_N} vs {expected_EN}"
    print("  -> Logarithmic negativity E_N = 2r verified across all r to < 1e-10 error (PASS)")

    # -----------------------------------------------------------------
    # Test 4: Physicality Boundary Detection
    # -----------------------------------------------------------------
    print("\nTest 4: Verifying Heisenberg-Robertson physicality boundary...")
    # Unphysical matrix: variance below vacuum in both quadratures
    V_unphysical = 0.25 * np.eye(2)
    is_phys = GaussianStateEngine.is_physical(V_unphysical)
    assert not is_phys, "Engine accepted unphysical sub-vacuum covariance matrix!"
    print("  -> Unphysical covariance matrix correctly rejected (PASS)")

    # -----------------------------------------------------------------
    # Test 5: von Neumann Entropy of Squeezed Thermal State
    # -----------------------------------------------------------------
    print("\nTest 5: Verifying entropy invariance under unitary squeezing...")
    n_th = 0.75
    _, V_th = GaussianStateEngine.thermal_state(1, n_th)
    S_th = GaussianStateEngine.von_neumann_entropy(V_th)
    
    # Squeeze the thermal state
    S1 = GaussianStateEngine.single_mode_squeezer(1.4)
    V_th_sq = S1 @ V_th @ S1.T
    S_th_sq = GaussianStateEngine.von_neumann_entropy(V_th_sq)
    
    assert np.isclose(S_th, S_th_sq, atol=1e-12), f"Entropy changed under squeezing: {S_th} vs {S_th_sq}"
    print(f"  -> Entropy invariance S(V) = {S_th:.6f} == S(S V S^T) = {S_th_sq:.6f} (PASS)")

    print("=" * 70)
    print("ALL 5 BENCHMARK TESTS PASSED WITH ZERO TOLERANCE DEFECTS!")
    print("=" * 70)

run_benchmark_suite()


---

## Summary & Pedagogical Bridge to Notebook 5.6

### Key Takeaways from Notebook 5.5:
1. **Mathematical Economy of Gaussian States**: Completely characterized by first moments $\vec{d}$ and covariance matrix $\mathbf{V}$, eliminating the need for infinite-dimensional Fock representations.
2. **Symplectic Symmetry**: Canonical transformations preserving $[\hat{R}_j, \hat{R}_k] = i\Omega_{jk}$ correspond to $\mathbf{S}\mathbf{\Omega}\mathbf{S}^T = \mathbf{\Omega}$, forming the group $\text{Sp}(2N, \mathbb{R})$.
3. **Williamson's Theorem**: Any valid covariance matrix is diagonalized by a symplectic transformation into an uncorrelated product of thermal states. The symplectic eigenvalues $\nu_k \ge 1/2$ measure intrinsic quantum mixedness invariant under passive and active linear optics.
4. **Entanglement Quantifier**: Continuous-variable entanglement is completely captured by the symplectic spectrum of the partially transposed covariance matrix $\tilde{\mathbf{V}}$, yielding the exact identity $E_N = 2r$ for pure two-mode squeezed states.

---

### Looking Ahead to Notebook 5.6: Quantum State Engineering & Photon Subtraction
While Gaussian states and linear optical operations are analytically tractable, **Gaussian operations alone cannot achieve universal quantum computation** (Gottesman-Knill theorem for CV) nor can they generate states with negative Wigner functions (Hudson's theorem). 

In **Notebook 5.6**, we venture beyond the Gaussian realm into **Quantum State Engineering**:
- Non-Gaussian conditional operations via photon addition ($\hat{a}^\dagger$) and photon subtraction ($\hat{a}$).
- Generating optical Schrödinger cat states by single-photon subtraction on squeezed vacuum: $\hat{a}\hat{S}(r)|0\rangle \approx |\mathrm{cat}_{\mathrm{odd}}\rangle$.
- Experimental modeling with weakly reflective beam splitters and avalanche photodiode (APD) threshold detectors.
- Quantifying the volume of Wigner negativity $\delta_W = \iint_{W<0} |W(x, p)| dx dp$ as a non-classical computational resource.
